In [1]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import Ollama
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from ragas import EvaluationDataset, evaluate
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextPrecisionWithoutReference,
)
from ragas.metrics import ContextRecall, FactualCorrectness
from langchain_community.chat_models import ChatOllama
import pandas as pd
import warnings

c:\terraria-chatbot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\lolmo\AppData\Local\Temp\ipykernel_15736\2176439336.py:9: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\Users\lolmo\AppData\Local\Temp\ipykernel_15736\2176439336.py:9: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ResponseRelevancy
  from ragas.metrics import (
C:\Users\lolmo\AppData\Local\Temp\ipykernel_15736\2176439336.py:9: DeprecationWarning: Importing LLMContextPrecisionWi

In [2]:
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

vector_base = Chroma(
    persist_directory="./chroma_terraria_db",
    embedding_function=embeddings
)

llm = Ollama(model='phi3')

system_prompt = (
    "You are an expert Terraria assistant. Your ONLY job is to extract answers from the provided CONTEXT.\n"
    "CRITICAL RULES:\n"
    "1. If the answer is not explicitly written in the CONTEXT below, you MUST reply exactly with: "
    "'I cannot answer this based on the provided text.'\n"
    "2. Do NOT invent items, bosses, or mechanics. Do NOT use your outside knowledge.\n"
    "3. Base your answer ONLY on the text below.\n\n"
    "CONTEXT:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

retriever = vector_base.as_retriever(search_kwargs={"k": 4})
qa_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, qa_chain)

print("RAG pipeline gotowy.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6221.66it/s]
C:\Users\lolmo\AppData\Local\Temp\ipykernel_15736\3510920083.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_base = Chroma(


RAG pipeline gotowy.


C:\Users\lolmo\AppData\Local\Temp\ipykernel_15736\3510920083.py:8: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model='phi3')


In [3]:
test_cases = [
    {
        "question": "What materials are needed to craft a Workbench?",
        "reference": "A Workbench requires 10 Wood to craft."
    },
    {
        "question": "How do you summon the Eye of Cthulhu?",
        "reference": "The Eye of Cthulhu can be summoned using a Suspicious Looking Eye at night, or it spawns automatically once certain conditions are met."
    },
    {
        "question": "What is the Corruption biome?",
        "reference": "The Corruption is an evil biome characterized by purple grass, Ebonstone, and flying enemies like Eaters of Souls."
    },
    {
        "question": "What does the Arms Dealer NPC sell?",
        "reference": "The Arms Dealer sells guns and ammunition, including the Minishark and Musket Balls."
    },
    {
        "question": "How do you enter Hardmode?",
        "reference": "Hardmode is activated by defeating the Wall of Flesh in the Underworld."
    },
    {
        "question": "How do you craft a Mana Crystal?",
        "reference": "A Mana Crystal is crafted by combining 3 Fallen Stars."
    },
    {
        "question": "Where can you find the Goblin Tinkerer?",
        "reference": "The Goblin Tinkerer can be found bound underground in the Caverns layer after a Goblin Army has been defeated."
    },
    {
        "question": "What does the Guide Voodoo Doll do?",
        "reference": "Throwing a Guide Voodoo Doll into lava in the Underworld kills the Guide and summons the Wall of Flesh boss."
    },
    {
        "question": "How do you craft a Gold Pickaxe?",
        "reference": "A Gold Pickaxe is crafted using 10 Gold Bars and 4 Wood at an Iron or Lead Anvil."
    },
    {
        "question": "What are the requirements for an NPC to move into a house?",
        "reference": "An NPC needs an empty house with background walls, a light source, a flat surface item (like a table), a comfort item (like a chair), and a door."
    },
    {
        "question": "Where do you find Hellstone?",
        "reference": "Hellstone is found in the Underworld and requires a pickaxe with at least 65% pickaxe power (like a Nightmare Pickaxe) to mine."
    },
    {
        "question": "What is the function of the Magic Mirror?",
        "reference": "The Magic Mirror is an item that teleports the player back to their designated spawn point when used."
    },
    {
        "question": "How do you summon King Slime?",
        "reference": "King Slime can be summoned manually using a Slime Crown, or he may spawn randomly during a Slime Rain event."
    },
    {
        "question": "What is the Zenith?",
        "reference": "The Zenith is a powerful, post-Moon Lord melee weapon crafted using a variety of specific swords obtained throughout the game's progression."
    },
    {
        "question": "How do you stop the spread of Corruption or Crimson?",
        "reference": "The spread can be stopped using the Clentaminator, planting Sunflowers, or digging quarantine trenches at least 3 blocks wide."
    },
    {
        "question": "What items does the Dryad NPC sell?",
        "reference": "The Dryad sells nature-themed items, including Purification Powder, Vile/Vicious Powder, grass seeds, and planter boxes."
    },
    {
        "question": "Where do you get the Muramasa sword?",
        "reference": "The Muramasa is found inside Locked Gold Chests located in the Dungeon."
    },
    {
        "question": "How do you defeat Skeletron?",
        "reference": "Skeletron must be summoned by talking to the Old Man at the Dungeon entrance at night. Destroying his hands first makes his head take significantly more damage."
    },
    {
        "question": "What causes a Meteorite biome to spawn?",
        "reference": "A Meteorite biome is created randomly when a meteor crashes into the world, usually after the player defeats the Eater of Worlds or Brain of Cthulhu."
    },
    {
        "question": "How do you craft a basic Grappling Hook?",
        "reference": "A basic Grappling Hook can be crafted using 3 Iron Chains or Lead Chains and 1 Hook at an Iron or Lead Anvil."
    }
]

In [4]:
results = []

for i, tc in enumerate(test_cases[:10]):
    question = tc["question"]
    print(f"Question {i+1}")

    rag_output = rag_chain.invoke({"input": question})

    contexts = [doc.page_content for doc in rag_output["context"]]
    answer = rag_output["answer"]

    results.append({
        "question": question,
        "answer": answer,
        "contexts": contexts,
        "reference": tc.get("reference"),
    })

Question 1
Question 2
Question 3
Question 4
Question 5
Question 6
Question 7
Question 8
Question 9
Question 10


### RAGAS

In [5]:
from openai import OpenAI as OpenAIClient
from ragas.llms import llm_factory

ollama_client = OpenAIClient(
    api_key="ollama",
    base_url="http://localhost:11434/v1"
)
ragas_llm = llm_factory("phi3", provider="openai", client=ollama_client)

from ragas.embeddings import LangchainEmbeddingsWrapper
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)


C:\Users\lolmo\AppData\Local\Temp\ipykernel_15736\3709095728.py:11: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)


In [ ]:
samples = [
    SingleTurnSample(
        user_input=r["question"],
        response=r["answer"],
        retrieved_contexts=r["contexts"],
        reference=r["reference"],
    )
    for r in results
]

dataset = EvaluationDataset(samples=samples)

metrics = [
    Faithfulness(llm=ragas_llm),
    ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings),
    LLMContextPrecisionWithoutReference(llm=ragas_llm),
    ContextRecall(llm=ragas_llm),
    FactualCorrectness(llm=ragas_llm),
]

evaluation_results = evaluate(
    dataset=dataset,
    metrics=metrics,
)

evaluation_results

Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[10]: InstructorRetryException(<failed_attempts>

<generation number="1">
<exception>
    4 validation errors for NLIStatementOutput
statements.0.verdict
  Field required [type=missing, input_value={'statement': 'The provid...s nature or mechanics.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
statements.1.verdict
  Field required [type=missing, input_value={'statement': 'The provid...s nature or mechanics.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
statements.2.verdict
  Field required [type=missing, input_value={'statement': 'The provid...s nature or mechanics.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
statements.3.verdict
  Field required [ty

In [ ]:
df_results = evaluation_results.to_pandas()
df_results["question"] = [r["question"] for r in results]

score_cols = [c for c in df_results.columns if c not in ("user_input", "response", "retrieved_contexts", "reference", "question")]

for col in score_cols:
    mean_val = df_results[col].mean()
    print(f"{col:<45} {mean_val:.3f}")

display(df_results[["question"] + score_cols].round(3))

,question,faithfulness,answer_relevancy,llm_context_precision_without_reference,question
0,What materials are needed to craft a Workbench?,0.333,0.923,0.806,What materials are needed to craft a Workbench?
1,How do you summon the Eye of Cthulhu?,0.667,0.000,0.000,How do you summon the Eye of Cthulhu?
2,What is the Corruption biome?,0.000,0.000,0.000,What is the Corruption biome?
3,Tell me about weapons in Terraria,1.000,0.788,0.250,Tell me about weapons in Terraria
4,What does the Arms Dealer NPC sell?,1.000,0.437,0.806,What does the Arms Dealer NPC sell?
5,How do you enter Hardmode?,1.000,0.820,0.500,How do you enter Hardmode?
6,What is the best recipe for a sword?,0.500,-0.000,0.000,What is the best recipe for a sword?
7,How do trees work in Terraria?,1.000,0.628,1.000,How do trees work in Terraria?


### Bad answers

### Save

In [ ]:
output_path = "./ragas_results.csv"
df_results.to_csv(output_path, index=False)

Wyniki zapisane do: ./ragas_results.csv
Wyniki z ground-truth zapisane do: ./ragas_results_with_reference.csv
